[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/11_grpo.ipynb)

# GRPO — reward-function RL

> **Fine-tuning series — 11 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 7b. RL — GRPO (reward-function RL)  ·  *what-axis*

When you can *score* an output but don't have preference pairs. The model samples
several completions per prompt; your `reward_funcs` rate them; it reinforces the
better ones relative to the group. This is the modern recipe for reasoning models.
Heavier than DPO (it generates during training).

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Reward = prefer short, lowercase answers (toy). Real ones check correctness/format.
def reward_brevity(completions, **kwargs):
    return [max(0.0, 1.0 - len(c) / 100.0) for c in completions]

grpo_ds = Dataset.from_list([{"prompt": e["instruction"]} for e in instructions])

grpo_trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=reward_brevity,
    train_dataset=grpo_ds,
    args=GRPOConfig(output_dir="ft_grpo", per_device_train_batch_size=2,
                    num_generations=2,          # completions sampled per prompt
                    max_completion_length=32, max_steps=10,
                    learning_rate=1e-6, logging_steps=5,
                    bf16=BF16_OK, report_to="none"),
)
grpo_trainer.train()
del grpo_trainer; cleanup()
# PPO (PPOTrainer) also exists — needs a separate reward model + value head; heaviest.

### Where this comes from: classic RLHF with PPO

Before DPO and GRPO, alignment meant the original **RLHF** recipe — three stages:

1. **SFT** — instruction-tune a base model (notebook 08).
2. **Reward model** — train a scalar model to score outputs so *chosen > rejected* (the reward
   modeling cell in notebook 13).
3. **PPO** — use that reward to do reinforcement learning, with a **KL penalty** keeping the
   policy close to the SFT model so it doesn't drift into gibberish.

PPO needs **four models in memory at once** (policy, reference, reward, value/critic) — heavy and
fiddly to stabilize. That cost is exactly why the field moved on: **DPO** drops the reward model
and RL entirely (notebook 10); **GRPO** keeps a reward *function* but drops PPO's critic by scoring
a *group* of samples relative to each other (above). Same goal — prefer better outputs — at a
fraction of the machinery.

In [ ]:
# Sketch of the PPO step (TRL). Shown for completeness — prefer DPO/GRPO above in practice.
try:
    from trl import PPOTrainer, PPOConfig
except ImportError:
    from trl.experimental.ppo import PPOTrainer, PPOConfig
# A PPOTrainer run wires together: the SFT policy model, a frozen reference model,
# the trained reward model (notebook 13), and a value head; then loops:
#   generate -> score with reward model -> ppo_trainer.step(queries, responses, rewards)
# It is the heaviest preference method; included here so the lineage is complete.